# 後処理・閾値最適化・順位最適化

評価指標に合わせて予測を後処理し、閾値や順位を最適化してスコアを底上げする方法を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/advanced/04-post-processing`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# 陽性率 15% の不均衡な検証ラベル y_valid と OOF 予測確率を用意します
import numpy as np

rng = np.random.default_rng(42)
n_samples = 1000
y_valid = (rng.random(n_samples) < 0.15).astype(int)
oof = np.where(y_valid == 1, rng.beta(2.6, 3.8, size=n_samples), rng.beta(1.1, 10.5, size=n_samples))

## 後処理で「指標に寄せる」

モデルの生の予測（確率など）を、**コンペの評価指標が最大になるよう加工**するのが後処理です。
モデルを変えずにスコアを底上げできることがあります。

## 閾値最適化（分類）

F1 などラベルで決まる指標では、確率を 0/1 にする**閾値**が重要です。
既定の 0.5 が最適とは限らないため、OOF 予測で最適な閾値を探します。

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

best_t, best_f1 = 0.5, 0
for t in np.linspace(0.05, 0.95, 91):
    f1 = f1_score(y_valid, (oof > t).astype(int))
    if f1 > best_f1:
        best_t, best_f1 = t, f1
# best_t を test 予測に適用する

In [ ]:
default_f1 = f1_score(y_valid, (oof > 0.5).astype(int))
print(f"Default threshold (0.50) F1: {default_f1:.4f} -> Optimized threshold ({best_t:.2f}) F1: {best_f1:.4f}")

> **Warning:** 閾値は必ず **OOF（検証側）** で決めます。テストや Public LB に合わせて閾値を動かすと、 LB へ過学習してシェイクアップの原因になります。

## 順位最適化・分布合わせ

- AUC のように**順位**で決まる指標では、確率の値そのものより順位が重要（ランク変換が有効）
- 目的変数の分布が既知なら、予測の分布を**実際の比率に合わせる**（キャリブレーション/補正）と効くことがある

## 制約のある後処理

問題に**制約**がある場合（合計が一定、各クラスの個数が既知など）、
その制約を満たすように予測を調整すると大きく効くことがあります。

> **Tip:** 後処理は「指標と問題設定を深く読む」ほど効きます。Overview と評価式を読み込み、 「何を最適化しているのか」を後処理に反映しましょう。ただし効果は必ず CV で確認します。

## 次のステップ

次は、データリークの発見と（ルール内での）扱い、外部データの注意点を学びます。